# 1. Power Data

In [1]:
import pandas as pd
import numpy as np
import sys
import os

from datetime import timedelta, datetime

In [3]:

xls_folder = './Raw Data/SEMS'


xls_files = [f for f in os.listdir(xls_folder) if f.endswith('.xls')]


for f in xls_files:
    print(f)

01_2024_Elektrárna_20250706182636.xls
01_2025_Elektrárna_20250706183210.xls
02_2024_Elektrárna_20250706182642.xls
02_2025_Elektrárna_20250706183229.xls
03_2024_Elektrárna_20250706182648.xls
03_2025_Elektrárna_20250706183242.xls
04_2024_Elektrárna_20250706182656.xls
04_2025_Elektrárna_20250706183251.xls
05_2024_Elektrárna_20250706182702.xls
05_2025_Elektrárna_20250706183258.xls
06_2024_Elektrárna_20250706182710.xls
06_2025_Elektrárna_20250706183313.xls
07_2024_Elektrárna_20250706182805.xls
07_2025_Elektrárna_20250805235627.xls
08_2024_Elektrárna_20250706182811.xls
09_2023_Elektrárna_20250706182010.xls
09_2024_Elektrárna_20250706183030.xls
10_2023_Elektrárna_20250706182412.xls
10_2024_Elektrárna_20250706183059.xls
11_2023_Elektrárna_20250706182426.xls
11_2024_Elektrárna_20250706183105.xls
12_2023_Elektrárna_20250706182434.xls
12_2024_Elektrárna_20250706183111.xls


In [4]:
df_raw = pd.DataFrame()

In [5]:

for file in xls_files:
    full_path = os.path.join(xls_folder, file)
    df_temp = pd.read_excel(full_path)
    df_temp['Unnamed: 0'] = df_temp['Unnamed: 0'].astype(str)
    df_temp = df_temp[~df_temp['Unnamed: 0'].str.lower().isin(['nan', 'měsíční zpráva', 'total'])]
    df_temp
    df_temp.columns = df_temp.iloc[0]
    df_temp = df_temp[1:].reset_index(drop = True)
    
    df_raw = pd.concat([df_raw, df_temp], ignore_index=True)

In [6]:
df_raw.to_csv(r'.\Model\SEMS_data.csv', index=False)

# 2. Weather Data

In [7]:
import requests


In [8]:
url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/Vala%C5%A1sk%C3%A9%20Mezi%C5%99%C3%AD%C4%8D%C3%AD/2023-09-01/2025-07-31?unitGroup=metric&elements=datetime%2Ctempmax%2Ctempmin%2Ctemp%2Cfeelslikemax%2Cfeelslikemin%2Cfeelslike%2Cdew%2Chumidity%2Cprecip%2Cprecipcover%2Cpreciptype%2Csnow%2Csnowdepth%2Cwindgust%2Cwindspeed%2Cwindspeedmax%2Cwindspeedmean%2Cwinddir%2Cpressure%2Ccloudcover%2Cvisibility%2Csolarradiation%2Csolarenergy%2Cuvindex%2Csevererisk%2Csunrise%2Csunset&include=obs%2Cremote%2Cstats%2Cdays&key=H5X3D8N2WKAAQ68QDPHD5UA8J&contentType=json"


In [9]:
response = requests.get(url)
data = response.json()


In [10]:
days = data.get("days", [])
df = pd.json_normalize(days)

In [11]:
df.to_csv("Model/weather.csv", index=False)

# 3. Predictions data


In [12]:
csv_folder = './Raw Data/Saved Predictions'


csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]


for f in csv_files :
    print(f)

predikce_pv_20250713_20250726.csv
pv_prediction_20250714_20250727.csv
pv_prediction_20250715_20250728.csv
pv_prediction_20250716_20250729.csv
pv_prediction_20250717_20250730.csv
pv_prediction_20250718_20250731.csv
pv_prediction_20250719_20250801.csv
pv_prediction_20250720_20250802.csv
pv_prediction_20250721_20250803.csv
pv_prediction_20250722_20250804.csv
pv_prediction_20250731_20250813.csv


In [13]:
df_raw = pd.DataFrame()

In [14]:

for file in csv_files:
    full_path = os.path.join(csv_folder, file)
    df_temp = pd.read_csv(full_path,usecols= ['datetime','PV(kWh)_pred'])
    df_temp['datetime'] = pd.to_datetime(df_temp['datetime'])
    df_raw = pd.concat([df_raw, df_temp], ignore_index=True)


In [15]:
df_raw

,datetime,PV(kWh)_pred
0,2025-07-13,42.005863
1,2025-07-14,43.352434
2,2025-07-15,31.931366
3,2025-07-16,30.425017
4,2025-07-17,28.925309
...,...,...
149,2025-08-09,38.562614
150,2025-08-10,45.983128
151,2025-08-11,42.248796
152,2025-08-12,44.141645


In [16]:
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'])

In [17]:
is_new_block = (df_raw['datetime'] <= df_raw['datetime'].shift(1))
df_raw['BlockId'] = is_new_block.cumsum()

start_date = datetime(2025, 7, 12)
df_raw["DownloadDate"] = df_raw["BlockId"].apply(lambda x: start_date + timedelta(days=x))
df_raw = df_raw.drop(columns=['BlockId'])

In [18]:
df_raw.head(5)

,datetime,PV(kWh)_pred,DownloadDate
0,2025-07-13,42.005863,2025-07-12
1,2025-07-14,43.352434,2025-07-12
2,2025-07-15,31.931366,2025-07-12
3,2025-07-16,30.425017,2025-07-12
4,2025-07-17,28.925309,2025-07-12


In [19]:
df_raw.to_csv("Model/prediction_history.csv", index=False)